# PARTE 2 — Búsquedas Prácticas en Azure AI Search


In [6]:
%pip install azure-search-documents --pre
%pip install azure-identity
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Aseguramos la conexión

In [45]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.models import VectorizableTextQuery

# 1. Cargar configuración forzando la actualización desde el .env
load_dotenv(override=True)

endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
key = os.getenv("AZURE_SEARCH_ADMIN_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX")
semantic_config = os.getenv("AZURE_SEARCH_SEMANTIC_CONFIG")

print(f"Intentando conectar a: {endpoint}...")

try:
    # 2. Probar conexión básica al servicio
    index_client = SearchIndexClient(endpoint, AzureKeyCredential(key))
    index_def = index_client.get_index(index_name)
    
    print("✅ ¡CONEXIÓN EXITOSA!")
    print(f"--- Configuración Semántica en uso: '{semantic_config}' ---")
    
    # 3. Inicializar el cliente de búsqueda
    client = SearchClient(endpoint, index_name, AzureKeyCredential(key))
    count = client.get_document_count()
    print(f"Documentos indexados: {count}")

    # 4. Definir función para imprimir resultados
    def print_results(results, title):
        print(f"\n{'='*20} {title} {'='*20}")
        for i, result in enumerate(results, 1):
            score = result.get('@search.score', 0)
            reranker = result.get('@search.reranker_score', 'N/A')
            content = result.get('chunk', 'Campo no encontrado')[:150]
            print(f"{i}. [Score: {score}] [Reranker: {reranker}]\n   Texto: {content}...\n")
            if i == 5: break

    # 5. Preparar la consulta
    query_text = "¿Qué información relevante contienen los documentos sobre el proyecto?"
    
    # Usamos 'text_vector' porque es el que detectamos en tu índice
    vector_query = VectorizableTextQuery(
        text=query_text, 
        k_nearest_neighbors=5, 
        fields="text_vector" 
    )
except Exception as e:
    print(f"\n❌ HA OCURRIDO UN ERROR:")
    print(f"Detalle: {e}")

Intentando conectar a: https://generativasearch.search.windows.net...


k_nearest_neighbors is not a known attribute of class <class 'azure.search.documents._generated.models._models_py3.VectorizableTextQuery'> and will be ignored


✅ ¡CONEXIÓN EXITOSA!
--- Configuración Semántica en uso: 'rag-1776328824836-semantic-configuration' ---
Documentos indexados: 15


In [46]:
def print_results(results, title, query):
    print(f"\n{'='*15} {title} {'='*15}")
    print(f"🔍 Consulta: '{query}'\n") # Imprimimos la consulta aquí
    
    for i, result in enumerate(results, 1):
        score = result.get('@search.score', 0)
        reranker = result.get('@search.reranker_score', 'N/A')
        content = result.get('chunk', 'Sin contenido')[:250].replace('\n', ' ') 
        
        print(f"🔸 RESULTADO {i}")
        print(f"   📊 Score Base: {score:.4f} | 🧠 Reranker IA: {reranker}")
        print(f"   📄 Texto: {content}...\n")
        
        if i == 3: break

In [47]:
query_text = "¿Cómo puedo combinar CASE WHEN y CAST para concatenar y formatear textos?"

# Limitamos la búsqueda matemática a 3 vecinos también
vector_query = VectorizableTextQuery(
    text=query_text, 
    k_nearest_neighbors=3, 
    fields="text_vector"
)

k_nearest_neighbors is not a known attribute of class <class 'azure.search.documents._generated.models._models_py3.VectorizableTextQuery'> and will be ignored


## Vector Search

In [54]:
res_vector = client.search(
    search_text=None, 
    vector_queries=[vector_query],
    select=["chunk", "title"],
    top=3
)
print_results(res_vector, "VECTOR SEARCH PURA", query_text)


=============== VECTOR SEARCH PURA ===============
🔍 Consulta: '¿Cómo puedo combinar CASE WHEN y CAST para concatenar y formatear textos?'

🔸 RESULTADO 1
   📊 Score Base: 0.7425 | 🧠 Reranker IA: None
   📄 Texto: AS DECIMAL(10,2)) AS precio_numerico,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 0.21, 2) AS iva_calculado,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 1.21, 2) AS precio_co...

🔸 RESULTADO 2
   📊 Score Base: 0.7418 | 🧠 Reranker IA: None
   📄 Texto: CASE WHEN + CAST — Guía de estudio para examen (tienda_online) Documento enfocado y autocontenido con definiciones, sintaxis, ejemplos resueltos y ejercicios con soluciones. Ideal para repasar antes del examen.  Visión rápida  CASE WHEN: expresión co...

🔸 RESULTADO 3
   📊 Score Base: 0.7387 | 🧠 Reranker IA: None
   📄 Texto: '$', ''), ',', '') AS DECIMAL(10,2)) * 0.21, 2) AS iva_calculado,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DE

### Hybrid Search

In [55]:
res_hybrid = client.search(
    search_text=query_text, 
    vector_queries=[vector_query],
    select=["chunk", "title"],
    top=3
)
print_results(res_hybrid, "HYBRID SEARCH (Vector + Keyword)", query_text)


=============== HYBRID SEARCH (Vector + Keyword) ===============
🔍 Consulta: '¿Cómo puedo combinar CASE WHEN y CAST para concatenar y formatear textos?'

🔸 RESULTADO 1
   📊 Score Base: 0.0333 | 🧠 Reranker IA: None
   📄 Texto: AS DECIMAL(10,2)) AS precio_numerico,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 0.21, 2) AS iva_calculado,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 1.21, 2) AS precio_co...

🔸 RESULTADO 2
   📊 Score Base: 0.0323 | 🧠 Reranker IA: None
   📄 Texto: CASE WHEN + CAST — Guía de estudio para examen (tienda_online) Documento enfocado y autocontenido con definiciones, sintaxis, ejemplos resueltos y ejercicios con soluciones. Ideal para repasar antes del examen.  Visión rápida  CASE WHEN: expresión co...

🔸 RESULTADO 3
   📊 Score Base: 0.0323 | 🧠 Reranker IA: None
   📄 Texto: '$', ''), ',', '') AS DECIMAL(10,2)) * 0.21, 2) AS iva_calculado,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), 

### Semantic Search

In [56]:
res_semantic = client.search(
    search_text=query_text,
    query_type="semantic",
    semantic_configuration_name=semantic_config,
    select=["chunk", "title"],
    top=3
)
print_results(res_semantic, "SEMANTIC SEARCH (Solo Texto)", query_text)


=============== SEMANTIC SEARCH (Solo Texto) ===============
🔍 Consulta: '¿Cómo puedo combinar CASE WHEN y CAST para concatenar y formatear textos?'

🔸 RESULTADO 1
   📊 Score Base: 7.3221 | 🧠 Reranker IA: 3.0946006774902344
   📄 Texto: AS DECIMAL(10,2)) AS precio_numerico,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 0.21, 2) AS iva_calculado,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 1.21, 2) AS precio_co...

🔸 RESULTADO 2
   📊 Score Base: 3.7839 | 🧠 Reranker IA: 2.989677906036377
   📄 Texto: '$', ''), ',', '') AS DECIMAL(10,2)) * 0.21, 2) AS iva_calculado,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 1.21, 2) AS precio_con_iva FROM practica_cast.datos_mixtos;  2) Extraer mes-año y truncar monto en tra...

🔸 RESULTADO 3
   📊 Score Base: 2.4068 | 🧠 Reranker IA: 2.801100254058838
   📄 Texto: tienda_online.productos ORDER BY precio DESC;  Explicación: ordenar las condiciones

### 4. Semantic Hybrid Search
El método más avanzado: híbrido + re-ranker semántico.

In [58]:
# D. SEMANTIC HYBRID SEARCH
res_sem_hybrid = client.search(
    search_text=query_text,
    vector_queries=[vector_query],
    query_type="semantic",
    semantic_configuration_name=semantic_config,
    select=["chunk", "title"],
    top=3
)
print_results(res_sem_hybrid, "SEMANTIC HYBRID SEARCH (La más precisa)", query_text)


=============== SEMANTIC HYBRID SEARCH (La más precisa) ===============
🔍 Consulta: '¿Cómo puedo combinar CASE WHEN y CAST para concatenar y formatear textos?'

🔸 RESULTADO 1
   📊 Score Base: 0.0333 | 🧠 Reranker IA: 3.0946006774902344
   📄 Texto: AS DECIMAL(10,2)) AS precio_numerico,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 0.21, 2) AS iva_calculado,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 1.21, 2) AS precio_co...

🔸 RESULTADO 2
   📊 Score Base: 0.0323 | 🧠 Reranker IA: 2.989677906036377
   📄 Texto: '$', ''), ',', '') AS DECIMAL(10,2)) * 0.21, 2) AS iva_calculado,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 1.21, 2) AS precio_con_iva FROM practica_cast.datos_mixtos;  2) Extraer mes-año y truncar monto en tra...

🔸 RESULTADO 3
   📊 Score Base: 0.0306 | 🧠 Reranker IA: 2.801100254058838
   📄 Texto: tienda_online.productos ORDER BY precio DESC;  Explicación: ordenar las 

In [59]:
query_text = "¿Qué opción me recomiendas si necesito mejorar la legibilidad de mi código y evitar recalcular un valor costoso, pero solo lo necesito de forma temporal?"

# Limitamos la búsqueda matemática a 3 vecinos también
vector_query = VectorizableTextQuery(
    text=query_text, 
    k_nearest_neighbors=3, 
    fields="text_vector"
)

k_nearest_neighbors is not a known attribute of class <class 'azure.search.documents._generated.models._models_py3.VectorizableTextQuery'> and will be ignored


In [60]:
res_vector = client.search(
    search_text=None, 
    vector_queries=[vector_query],
    select=["chunk", "title"],
    top=3
)
print_results(res_vector, "VECTOR SEARCH PURA", query_text)


=============== VECTOR SEARCH PURA ===============
🔍 Consulta: '¿Qué opción me recomiendas si necesito mejorar la legibilidad de mi código y evitar recalcular un valor costoso, pero solo lo necesito de forma temporal?'

🔸 RESULTADO 1
   📊 Score Base: 0.6319 | 🧠 Reranker IA: None
   📄 Texto: '$', ''), ',', '') AS DECIMAL(10,2)) * 0.21, 2) AS iva_calculado,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 1.21, 2) AS precio_con_iva FROM practica_cast.datos_mixtos;  2) Extraer mes-año y truncar monto en tra...

🔸 RESULTADO 2
   📊 Score Base: 0.6310 | 🧠 Reranker IA: None
   📄 Texto: que muestra sólo empleados del departamento IT. Ventaja: puedes dar permisos sólo sobre la vista y ocultar columnas sensibles (como salario a algunos roles).  Vista compleja / agregada — Ejemplo  CREATE VIEW salario_promedio AS SELECT d.nombre AS dep...

🔸 RESULTADO 3
   📊 Score Base: 0.6249 | 🧠 Reranker IA: None
   📄 Texto: tienda_online.productos ORDER BY precio DESC;  Explic

In [61]:
res_hybrid = client.search(
    search_text=query_text, 
    vector_queries=[vector_query],
    select=["chunk", "title"],
    top=3
)
print_results(res_hybrid, "HYBRID SEARCH (Vector + Keyword)", query_text)


=============== HYBRID SEARCH (Vector + Keyword) ===============
🔍 Consulta: '¿Qué opción me recomiendas si necesito mejorar la legibilidad de mi código y evitar recalcular un valor costoso, pero solo lo necesito de forma temporal?'

🔸 RESULTADO 1
   📊 Score Base: 0.0313 | 🧠 Reranker IA: None
   📄 Texto: - fecha_registro AS INTEGER) AS dias_como_cliente,   CASE     WHEN CAST(CURRENT_DATE - fecha_registro AS INTEGER) < 30 THEN 'Cliente Nuevo'     WHEN CAST(CURRENT_DATE - fecha_registro AS INTEGER) < 180 THEN 'Cliente Reciente'     WHEN CAST(CURRENT_DA...

🔸 RESULTADO 2
   📊 Score Base: 0.0312 | 🧠 Reranker IA: None
   📄 Texto: '$', ''), ',', '') AS DECIMAL(10,2)) * 0.21, 2) AS iva_calculado,   ROUND(CAST(REPLACE(REPLACE(precio_texto, '$', ''), ',', '') AS DECIMAL(10,2)) * 1.21, 2) AS precio_con_iva FROM practica_cast.datos_mixtos;  2) Extraer mes-año y truncar monto en tra...

🔸 RESULTADO 3
   📊 Score Base: 0.0312 | 🧠 Reranker IA: None
   📄 Texto: AS SELECT categoria_id, SUM(total) AS t

In [62]:
res_semantic = client.search(
    search_text=query_text,
    query_type="semantic",
    semantic_configuration_name=semantic_config,
    select=["chunk", "title"],
    top=3
)
print_results(res_semantic, "SEMANTIC SEARCH (Solo Texto)", query_text)


=============== SEMANTIC SEARCH (Solo Texto) ===============
🔍 Consulta: '¿Qué opción me recomiendas si necesito mejorar la legibilidad de mi código y evitar recalcular un valor costoso, pero solo lo necesito de forma temporal?'

🔸 RESULTADO 1
   📊 Score Base: 5.9254 | 🧠 Reranker IA: 2.1168861389160156
   📄 Texto: - fecha_registro AS INTEGER) AS dias_como_cliente,   CASE     WHEN CAST(CURRENT_DATE - fecha_registro AS INTEGER) < 30 THEN 'Cliente Nuevo'     WHEN CAST(CURRENT_DATE - fecha_registro AS INTEGER) < 180 THEN 'Cliente Reciente'     WHEN CAST(CURRENT_DA...

🔸 RESULTADO 2
   📊 Score Base: 6.3179 | 🧠 Reranker IA: 1.8128403425216675
   📄 Texto: Resumen Final — CTEs, Vistas, CASE WHEN y CAST Documento breve pensado para repasar los conceptos y ejemplos clave antes del examen. Cada ejemplo incluye una explicación paso a paso de cómo está construido y por qué funciona.  1) CTEs (WITH) — Qué, c...

🔸 RESULTADO 3
   📊 Score Base: 3.5326 | 🧠 Reranker IA: 1.6519112586975098
   📄 Texto: R

In [63]:
res_sem_hybrid = client.search(
    search_text=query_text,
    vector_queries=[vector_query],
    query_type="semantic",
    semantic_configuration_name=semantic_config,
    select=["chunk", "title"],
    top=3
)
print_results(res_sem_hybrid, "SEMANTIC HYBRID SEARCH (La más precisa)", query_text)


=============== SEMANTIC HYBRID SEARCH (La más precisa) ===============
🔍 Consulta: '¿Qué opción me recomiendas si necesito mejorar la legibilidad de mi código y evitar recalcular un valor costoso, pero solo lo necesito de forma temporal?'

🔸 RESULTADO 1
   📊 Score Base: 0.0313 | 🧠 Reranker IA: 2.1168861389160156
   📄 Texto: - fecha_registro AS INTEGER) AS dias_como_cliente,   CASE     WHEN CAST(CURRENT_DATE - fecha_registro AS INTEGER) < 30 THEN 'Cliente Nuevo'     WHEN CAST(CURRENT_DATE - fecha_registro AS INTEGER) < 180 THEN 'Cliente Reciente'     WHEN CAST(CURRENT_DA...

🔸 RESULTADO 2
   📊 Score Base: 0.0305 | 🧠 Reranker IA: 1.8128403425216675
   📄 Texto: Resumen Final — CTEs, Vistas, CASE WHEN y CAST Documento breve pensado para repasar los conceptos y ejemplos clave antes del examen. Cada ejemplo incluye una explicación paso a paso de cómo está construido y por qué funciona.  1) CTEs (WITH) — Qué, c...

🔸 RESULTADO 3
   📊 Score Base: 0.0293 | 🧠 Reranker IA: 1.6519112586975098
  

### Extra: Búsqueda con Scoring Profile
Si creaste un perfil llamado 'miPerfil' en el portal, aquí se aplica.

In [64]:
res_scoring = client.search(
    search_text=query_text,
    scoring_profile="miPerfil" # Cambia esto por el nombre que hayas puesto en el portal
)
print_results(res_scoring, "EXTRA: SCORING PROFILE", query_text)


=============== EXTRA: SCORING PROFILE ===============
🔍 Consulta: '¿Qué opción me recomiendas si necesito mejorar la legibilidad de mi código y evitar recalcular un valor costoso, pero solo lo necesito de forma temporal?'



HttpResponseError: (InvalidRequestParameter) Unknown scoring profile 'miPerfil'.
Parameter name: scoringProfile
Code: InvalidRequestParameter
Message: Unknown scoring profile 'miPerfil'.
Parameter name: scoringProfile
Exception Details:	(UnknownScoringProfile) Unknown scoring profile 'miPerfil'.
	Code: UnknownScoringProfile
	Message: Unknown scoring profile 'miPerfil'.